# Observatório dos Leilões Minerais — B2
## Do arremate aos royalties: dados, trajetórias, mapas e tempos

Este caderno contém os **scripts e o extrato analítico preservado dentro do próprio arquivo**. Abra no Google Colab e selecione **Ambiente de execução → Executar tudo**. Não exige login na ANM, caminhos do computador do autor ou montagem do Drive.

**Duas operações distintas**
1. **Reproduzir B2:** recalcula as tabelas e seis figuras a partir do extrato integrado preservado. Esse extrato contém decisões documentais e janelas já revistas; não é uma base bruta da ANM.
2. **Consultar a ANM:** baixa SOPLe, estoque, avaliação social, SCM e cinco blocos de CFEM diretamente da ANM. Reconstrói a rede direcional, recupera atos e recalcula CFEM e comercialização da coorte publicada. A comparação registra alterações sem substituir a versão do artigo. A coleta envolve cerca de 600 MB e pode levar vários minutos.

**Corte fiscal:** junho de 2026. **Cadastros da versão do artigo:** setembro de 2026. Valores nominais. Nominação, conciliação bancária e impacto causal não são inferidos da ausência de registros. A seleção social é um caminho próprio, sem lance financeiro.

Artigo, dados agregados e painel: https://observatorio-dos-leiloes-minerais.vercel.app/b2


In [ ]:
import sys, subprocess, importlib.util, os
if importlib.util.find_spec('matplotlib') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'matplotlib>=3.8,<4', 'numpy>=1.26,<3'])
from pathlib import Path
ROOT = Path.cwd() / 'observatorio_b2'
ROOT.mkdir(exist_ok=True)
CONSULTAR_ANM = True  # False: reproduz somente a versão preservada do artigo.


### 1. Recuperar o pacote preservado e conferir integridade
A célula contém um ZIP codificado. O resumo criptográfico confere se os arquivos reproduzidos são os mesmos da B2; não certifica o conteúdo das declarações dos regulados.

In [ ]:
import base64, zipfile, io, hashlib, json
PAYLOAD = '...'
raw = base64.b64decode(PAYLOAD)
assert hashlib.sha256(raw).hexdigest() == '8304958bafc8921e4ce5908d0d99421501249373692fbc2705ce2c0f5d4178e1'
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    for info in z.infolist():
        target = (ROOT / info.filename).resolve()
        assert target.is_relative_to(ROOT.resolve())
        z.extract(info, ROOT)
manifesto = json.loads((ROOT/'dados/manifesto_preservado.json').read_text())
for name, expected in manifesto.items():
    assert hashlib.sha256((ROOT/name).read_bytes()).hexdigest() == expected, name
print('Pacote B2 íntegro:', len(manifesto), 'arquivos verificados.')


### 2. Recalcular tabelas, figuras e arquivos por processo
O código calcula contagens sem duplicar por substância, janelas simétricas de 12 e 24 meses, tempos até o primeiro recolhimento, quartis, maturidade por rodada, grupos de acompanhamento, mapas e quantidades por substância e unidade. O código-fonte é extraído em `analise_b2.py` e pode ser aberto e alterado no explorador lateral do Colab.

In [ ]:
import importlib.util
def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    m = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(m)
    return m
analise = load_module('analise_b2', ROOT/'analise_b2.py')
metricas = analise.run()


### 3. Ler as figuras
Cinza no mapa de calor significa período ainda não observado, e não zero. As medianas descrevem apenas quem apresentou recolhimento. Não são previsões para todos os projetos nem comparação causal entre seleção social e leilão.

In [ ]:
from IPython.display import display, Image
for f in sorted((ROOT/'resultados').glob('figura*.png')):
    print(f.name)
    display(Image(filename=str(f), width=1000))


### 4. Conferir resultados estruturados
As tabelas CSV incluem identificadores de processos, períodos, substâncias e localidades. Quantidades em massa são harmonizadas quando compatíveis; toneladas e metros cúbicos nunca são somados. Os dados de comercialização são autodeclarados e não equivalem a medição independente da produção.

In [ ]:
import pandas as pd
for name in ['tabela_1_financeiro','tabela_2_transicoes','tabela_C3_regimes','tabela_C4_janelas','resumo_tempos','cfem_por_uf']:
    print(name)
    display(pd.read_csv(ROOT/'resultados'/f'{name}.csv'))


### 5. Baixar diretamente da ANM e reconstruir os vínculos
Fontes: [SOPLe](https://dadosabertos.anm.gov.br/SOPLE/), [Cadastro Mineiro](https://dadosabertos.anm.gov.br/SCM/microdados/) e [CFEM](https://dadosabertos.anm.gov.br/CFEM/). SIGMINE e malha estadual entram na reprodução cartográfica preservada; esta etapa não atualiza geometrias.

O coletor valida esquema, tamanho, integridade e data de acesso. Segue associações do original ao sucessor, conserva novos ciclos para conferência e exporta eventos, títulos e fases. Não trata arquivamento como retorno automático ao estoque. Mudanças de estrutura fazem a execução parar com diagnóstico.

In [ ]:
coleta = load_module('coleta_anm', ROOT/'coleta_anm.py')
if CONSULTAR_ANM:
    manifesto_atual = coleta.download(ROOT/'bases_atuais')
    relatorio = coleta.integrate(ROOT/'bases_atuais', ROOT/'dados/analitico_preservado.json', ROOT/'atualizacao')
    display(pd.DataFrame(relatorio['comparacoes']))
else:
    print('Coleta atual desativada. Resultados acima reproduzem o artigo B2.')


### 6. Como interpretar atualizações
A coorte do artigo permanece fixa para que mudanças nas bases não alterem silenciosamente sua conclusão. `vinculos_a_conferir.csv` permite revisar novas associações e desaparecimentos de vínculos. `nao_pagamento_evento_2464.csv` recupera evidências administrativas atuais. Os registros fiscais são reatribuídos somente às janelas documentadas; ambiguidades vão para arquivo próprio. O uso de janelas preservadas está explicitado no relatório — não significa refazer automaticamente todas as decisões documentais do artigo.

O extrato acompanha áreas e processos, não avalia a regularidade de pessoas. Datas de títulos não provam operação; uma guia publicada não certifica todas as condições de vigência. Comercialização acima de limite ou após um prazo é motivo de conferência, não prova automática de infração. Resultados de piloto com analistas somente podem ser incluídos com seus registros.

In [ ]:
import shutil
archive = shutil.make_archive(str(ROOT/'B2_resultados'), 'zip', ROOT/'resultados')
print('Resultados:', archive)
print('Scripts:', ROOT/'analise_b2.py', ROOT/'coleta_anm.py')
print('Atualização:', ROOT/'atualizacao')
# Para baixar no Colab, descomente:
# from google.colab import files
# files.download(archive)
